# Skip-gram from scratch — a configuration grid

Everything is in this notebook: tokeniser, vocabulary, the skip-gram model, the training loop
and the evaluation. No project files to mount, nothing to `pip install` beyond what Kaggle
already ships.

**Before running**

1. Upload `data/tinystories_20m.txt` (99 MB) as a Kaggle dataset and attach it to this notebook.
2. Internet is not needed once the dataset is attached.
3. Settings → Accelerator → GPU is optional: it makes the grid about five times faster, because
   the cost per step is the optimizer update, not the sampling.

Eight configurations, one variable changed at a time, each scored on the same four numbers.

## 0. Environment

In [ ]:
import json
import os
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

# torch prints a sparse-tensor notice on every optimiser step; it is not a
# problem and it drowns the output that matters.
warnings.filterwarnings("ignore", message="Sparse invariant checks")

SEED = 0
PAIRS_PER_RUN = 50_000_000   # what one "epoch" means here: 50M sampled pairs
BATCH = 8192                 # kept at 8192 on both devices so runs stay comparable
LR = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

print("torch", torch.__version__, "| device", DEVICE, "| cores", os.cpu_count())
if DEVICE.type == "cuda":
    print("gpu  ", torch.cuda.get_device_name(0))

## 1. The corpus

`tinystories_20m.txt` is 20,104,405 tokens of children's stories: simple grammar, small
vocabulary, and strong narrative structure. That is why a CPU can learn decent vectors in
minutes — and why the numbers below are comparable across runs.

The file is lowercased on load. `The` and `the` must be one word, or every count is split in
two.

In [ ]:
CORPUS_NAME = "tinystories_20m.txt"
DATASET = Path("/kaggle/input/datasets/elliyahurosha/tinystories-20m")


def find_corpus() -> Path:
    """Locate the corpus locally, or in the Kaggle dataset attached above."""
    for candidate in (Path("data") / CORPUS_NAME, Path(CORPUS_NAME)):
        if candidate.exists():
            return candidate
    hits = sorted(DATASET.rglob(CORPUS_NAME)) if DATASET.exists() else []
    if hits:
        return hits[0]
    raise FileNotFoundError(
        f"{CORPUS_NAME} not found in {DATASET}. "
        "Attach the dataset, or point DATASET at the folder you uploaded."
    )


corpus_path = find_corpus()
text = corpus_path.read_text(encoding="utf-8")
if not text.strip():
    raise ValueError(f"corpus is empty: {corpus_path}")

print(f"path  : {corpus_path}")
print(f"size  : {corpus_path.stat().st_size / 1e6:.0f} MB")
print(f"chars : {len(text):,}")
text = text.lower()
print()
print(text[:280])

## 2. Tokens, vocabulary, ids

Three steps, each with one decision worth knowing:

- **Tokenisation** keeps ASCII words with an optional internal apostrophe (`don't`). Numbers and
  punctuation are dropped on purpose — this is not a general tokeniser.
- **Vocabulary** keeps words seen at least 5 times; everything rarer becomes one `<unk>` id.
  Ordering by count then alphabetically makes the ids reproducible.
- **Encoding** turns the token list into an array of int32 ids: 4 bytes per token instead of a
  pointer per token.

In [ ]:
import re

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")
UNK = "<unk>"


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text)


def build_vocab(tokens: list[str], min_count: int = 5) -> dict[str, int]:
    """Id 0 is always UNK; the rest are ordered by frequency, then alphabetically."""
    counts = Counter(tokens)
    kept = [pair for pair in counts.items() if pair[1] >= min_count]
    kept.sort(key=lambda pair: (-pair[1], pair[0]))
    vocab = {UNK: 0}
    for word, _ in kept:
        vocab[word] = len(vocab)
    return vocab


def encode(tokens: list[str], vocab: dict[str, int]) -> np.ndarray:
    return np.fromiter(
        (vocab.get(token, 0) for token in tokens), dtype=np.int32, count=len(tokens)
    )


tokens = tokenize(text)
vocab = build_vocab(tokens)
ids = encode(tokens, vocab)
words = list(vocab)
unk_share = float((ids == 0).mean())
del tokens, text

print(f"tokens     : {len(ids):,}")
print(f"vocabulary : {len(vocab):,} words (min_count=5)")
print(f"UNK share  : {unk_share:.2%}")
print(f"ids array  : {ids.nbytes / 1e6:.0f} MB")

## 3. Training pairs and negatives

A 20M-token corpus has ~200M in-window pairs. Materialising them is pointless: skip-gram only
ever needs a batch. So each step draws random positions and a random offset in `1..window`.

**Subsampling** is the one algorithmic change this grid introduces. Frequent words carry almost
no signal (`the` predicts nothing) but occupy most of the window. Word2Vec drops each token with
probability `1 - sqrt(t / frequency)`, so a word 100x above the threshold keeps about a tenth of
its occurrences. The corpus shrinks; the signal per pair rises.

**Negatives** are drawn from counts raised to the 0.75 power, which flattens the head so rare
words are not ignored. `<unk>` gets weight zero: it is a bucket, not a word.

In [ ]:
def sample_pairs(ids, batch_size: int, window: int, rng: np.random.Generator):
    """Draw (center, context) pairs. Pairs touching UNK are rejected and redrawn."""
    centers = np.empty(batch_size, dtype=np.int32)
    contexts = np.empty(batch_size, dtype=np.int32)
    filled = 0
    while filled < batch_size:
        missing = batch_size - filled
        positions = rng.integers(window, len(ids) - window, size=missing)
        steps = rng.integers(1, window + 1, size=missing) * rng.choice([-1, 1], size=missing)
        context_positions = positions + steps
        keep = (ids[positions] != 0) & (ids[context_positions] != 0)
        taken = int(keep.sum())
        centers[filled:filled + taken] = ids[positions[keep]]
        contexts[filled:filled + taken] = ids[context_positions[keep]]
        filled += taken
    return centers, contexts


def subsample_ids(ids: np.ndarray, threshold: float, rng: np.random.Generator) -> np.ndarray:
    """Drop frequent tokens with probability 1 - sqrt(threshold / frequency)."""
    counts = np.bincount(ids, minlength=int(ids.max()) + 1)
    frequency = counts / counts.sum()
    keep = np.ones_like(frequency)
    common = frequency > threshold
    keep[common] = np.sqrt(threshold / frequency[common])
    keep[0] = 1.0
    return ids[rng.random(ids.size) < keep[ids]]


def negative_distribution(ids: np.ndarray, vocab_size: int, power: float = 0.75) -> np.ndarray:
    counts = np.bincount(ids, minlength=vocab_size).astype(np.float64)
    weights = counts**power
    weights[0] = 0.0
    return weights / weights.sum()


def sample_negatives(batch_size, num_negatives, probabilities, rng: np.random.Generator):
    cumulative = np.cumsum(probabilities)
    cumulative /= cumulative[-1]
    cumulative[-1] = 1.0
    draws = rng.random((batch_size, num_negatives))
    return np.searchsorted(cumulative, draws).astype(np.int32)


probabilities = negative_distribution(ids, len(vocab))
probe = np.random.default_rng(SEED)
print(f"UNK in 1M negative draws: {(np.searchsorted(np.cumsum(probabilities), probe.random(1_000_000)) == 0).sum()}")
print(f"top word share: {np.bincount(ids, minlength=len(vocab)).max() / len(ids):.2%}")

## 4. The model

Two embedding tables and a dot product. A word plays two roles — centre and context — and
sharing one vector for both mixes two different gradients. Word2Vec does the same; at the end we
keep the `center` table.

Initialisation matters more than it looks. PyTorch fills `nn.Embedding` with `N(0, 1)`, which
makes dot products of 100-dimensional vectors have a standard deviation around 10: `sigmoid`
saturates and the gradients are dead on arrival. The Word2Vec box `U(-0.5/d, 0.5/d)` starts every
score at zero, `sigmoid` at 0.5, and the loss at exactly `ln 2 = 0.693` — maximal ignorance.

In [ ]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size: int, dim: int, sparse: bool = True) -> None:
        super().__init__()
        # sparse=True keeps each step's gradient to the rows that were used, which
        # is what makes a CPU viable. CUDA has no sparse Adagrad kernel, so the GPU
        # path uses dense tables (11k x 300 floats is 27 MB -- noise on a GPU).
        self.center = nn.Embedding(vocab_size, dim, sparse=sparse)
        self.context = nn.Embedding(vocab_size, dim, sparse=sparse)
        bound = 0.5 / dim
        nn.init.uniform_(self.center.weight, -bound, bound)
        nn.init.uniform_(self.context.weight, -bound, bound)

    def forward(self, centers: torch.Tensor, contexts: torch.Tensor) -> torch.Tensor:
        """centers: (B,)  contexts: (B, K)  ->  scores: (B, K)"""
        return torch.einsum("bd,bkd->bk", self.center(centers), self.context(contexts))


torch.manual_seed(SEED)
untrained = SkipGram(len(vocab), 100)
print("parameters:", f"{sum(p.numel() for p in untrained.parameters()):,}")

## 5. Training

Five lines, repeated:

| line | what it does |
|---|---|
| sample real `(center, context)` pairs | what the corpus says |
| sample `K` random words per pair | what the corpus does not say |
| `logits = model(centers, contexts)` | one dot product per pair |
| `loss = BCEWithLogits(logits, targets)` | push positives up, negatives down |
| `backward()` + `step()` | nudge the numbers downhill |

`targets` is 1 in column 0 and 0 in columns 1..K. That is the whole supervision signal: no
labels, no dictionary, no human annotation.

**Adagrad, not Adam.** Adam normalises every coordinate to a step of about `lr`, so a small
shared bias accumulates linearly and every vector ends up pointing the same way: measured on this
data, PC1 held 97.6% of the variance and every pair had cosine ≈ 1.00. Adagrad's per-coordinate
accumulator keeps the space spread out (PC1 ≈ 0.06). It is not a detail; it is the difference
between vectors and noise.

In [ ]:
def train(config: dict, ids: np.ndarray, probabilities: np.ndarray, vocab_size: int):
    steps = config.get("steps") or int(config["epochs"] * PAIRS_PER_RUN / BATCH)
    rng = np.random.default_rng(SEED)
    torch.manual_seed(SEED)

    data = ids
    if config["subsample"] > 0:
        data = subsample_ids(ids, config["subsample"], rng)

    model = SkipGram(vocab_size, config["dim"], sparse=DEVICE.type == "cpu").to(DEVICE)
    optimizer = torch.optim.Adagrad(model.parameters(), lr=config.get("lr", LR))
    loss_fn = nn.BCEWithLogitsLoss()
    targets = torch.cat(
        [torch.ones(BATCH, 1), torch.zeros(BATCH, config["negatives"])], dim=1
    ).to(DEVICE)

    started = time.perf_counter()
    running, last = 0.0, float("nan")
    for step in range(1, steps + 1):
        centers, contexts = sample_pairs(data, BATCH, config["window"], rng)
        negatives = sample_negatives(BATCH, config["negatives"], probabilities, rng)
        logits = model(
            torch.as_tensor(centers, dtype=torch.long, device=DEVICE),
            torch.as_tensor(
                np.column_stack([contexts, negatives]), dtype=torch.long, device=DEVICE
            ),
        )
        loss = loss_fn(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running += float(loss.detach())
        if step % max(1, steps // 4) == 0:
            rate = step / (time.perf_counter() - started)
            print(
                f"  step {step:6,}/{steps:,}  loss {running / max(1, steps // 4):.4f}  "
                f"eta {(steps - step) / rate:4.0f}s",
                flush=True,
            )
            running, last = 0.0, float(loss.detach())

    elapsed = time.perf_counter() - started
    tokens = int(data.size)
    print(
        f"  {steps:,} steps, {steps * BATCH / 1e6:,.1f}M pairs in {elapsed:.0f}s "
        f"({steps * BATCH / elapsed / 1000:.0f}k pairs/s), "
        f"{len(ids) / tokens:.1f}x fewer tokens",
        flush=True,
    )
    return {
        "loss": round(last, 4),
        "seconds": round(elapsed, 1),
        "steps": steps,
        "tokens_used": tokens,
    }, model.center.weight.detach().cpu().numpy()

## 6. Evaluation

Four numbers, because no single one is trustworthy:

- **baseline** — mean cosine of 5,000 unrelated pairs. Cosine values in an embedding space are
  compressed into a common direction, so absolute values mean nothing without this floor.
- **pc1** — variance share of the first principal component. Near 1.0 means the space collapsed.
- **group margin** — within-group cosine minus the all-pairs average over six hand-made groups
  (royalty, family, animals, places, feelings, size). The model was never told these groups exist.
- **analogies** — 16 questions answered by `b - a + c`, with top-1 accuracy and the median rank
  of the expected word. Median rank moves even when accuracy is flat.

In [ ]:
GROUPS = {
    "royalty": ["king", "queen", "prince", "princess"],
    "family": ["mom", "dad", "brother", "sister"],
    "animals": ["dog", "cat", "bird", "fish", "bear", "rabbit"],
    "places": ["castle", "forest", "garden", "house", "school"],
    "feelings": ["happy", "sad", "scared", "angry"],
    "size": ["big", "small"],
}

# a -> b is the relation; apply it to c and expect `expected`.
# Written from the language, not from what the model happens to get right.
ANALOGIES = [
    ("king", "queen", "prince", "princess"),
    ("king", "queen", "man", "woman"),
    ("boy", "girl", "man", "woman"),
    ("boy", "girl", "brother", "sister"),
    ("mom", "dad", "mother", "father"),
    ("mom", "dad", "sister", "brother"),
    ("son", "daughter", "brother", "sister"),
    ("cat", "cats", "dog", "dogs"),
    ("bird", "birds", "fish", "fishes"),
    ("walk", "walked", "jump", "jumped"),
    ("play", "played", "watch", "watched"),
    ("open", "opened", "close", "closed"),
    ("run", "ran", "eat", "ate"),
    ("big", "bigger", "small", "smaller"),
    ("good", "better", "bad", "worse"),
    ("day", "days", "night", "nights"),
]


def score(embedding: np.ndarray, words: list[str]) -> dict:
    unit = embedding / np.linalg.norm(embedding, axis=1, keepdims=True)
    index = {word: i for i, word in enumerate(words)}
    rng = np.random.default_rng(SEED)

    left = rng.choice(len(unit), 5000, replace=False)
    right = rng.choice(len(unit), 5000, replace=False)
    baseline = float((unit[left] * unit[right]).sum(axis=1).mean())

    centered = unit - unit.mean(axis=0, keepdims=True)
    spectrum = np.linalg.svd(centered, compute_uv=False)
    pc1 = float((spectrum**2 / (spectrum**2).sum())[0])

    picked = [index[w] for group in GROUPS.values() for w in group if w in index]
    block = unit[picked] @ unit[picked].T
    np.fill_diagonal(block, np.nan)
    inside, edge = [], 0
    for members in GROUPS.values():
        size = len(members)
        inside.append(np.nanmean(block[edge:edge + size, edge:edge + size]))
        edge += size
    margin = float(np.mean(inside) - np.nanmean(block))

    hits, ranks, used = 0, [], 0
    detail = []
    for a, b, c, expected in ANALOGIES:
        if not all(word in index for word in (a, b, c, expected)):
            continue
        used += 1
        query = unit[index[b]] - unit[index[a]] + unit[index[c]]
        scores = unit @ query
        scores[[index[a], index[b], index[c]]] = -np.inf
        gave = words[int(np.argmax(scores))]
        hits += gave == expected
        ranks.append(int((scores > scores[index[expected]]).sum()) + 1)
        detail.append({"question": f"{b} - {a} + {c}", "expected": expected, "gave": gave})

    neighbours = {}
    for group, members in GROUPS.items():
        for word in members:
            if word not in index:
                continue
            scores = unit @ unit[index[word]]
            best = np.argsort(-scores)[:6]
            neighbours[word] = [[words[i], round(float(scores[i]), 3)]
                                for i in best if i != index[word]][:5]

    return {
        "baseline": round(baseline, 3),
        "pc1": round(pc1, 3),
        "group_margin": round(margin, 3),
        "analogies": f"{hits}/{used}",
        "accuracy": round(hits / used, 3) if used else 0.0,
        "median_rank": float(np.median(ranks)) if ranks else None,
        "analogies_detail": detail,
        "neighbours": neighbours,
    }

## 7. Smoke test — 30 steps, before the grid

If the pipeline is broken, it is cheaper to find out here than twenty minutes in.

In [ ]:
smoke_config = {"name": "smoke", "dim": 100, "window": 5, "negatives": 5,
                "subsample": 1e-4, "epochs": 0, "steps": 30}
smoke_summary, smoke_embedding = train(smoke_config, ids, probabilities, len(vocab))
print("  loss 0.6931 at step 1 is ln 2, i.e. knows nothing; it must be lower now:",
      smoke_summary["loss"])
assert smoke_summary["loss"] < 0.69, "the loss did not move: something is wrong"

## 8. Round 2 — kill what lost, dig into what won

Round 1 ran nine configurations. It closed two doors and opened one:

| direction | measured | verdict |
|---|---|---|
| subsampling, `t = 1e-4` | margin 0.229 vs 0.259 at 50M pairs; 0.241 vs 0.282 at 150M | **dropped** — it removes real context, and this corpus is too small to need the head trimmed |
| more dimensions (200, 300) | margin 0.204 / 0.187, analogies 11/16 | **dropped** — 20M tokens do not support the parameters |
| window 2 | margin 0.283, analogies **15/16** | **keep digging** |
| negatives 10 | analogies 14/16 at window 5 | keep, retest at the winning window |
| five passes | 0.242 vs 0.241 — nothing | retest at the winning window |

That is what a grid is for: killing a direction is cheaper than arguing about it.

Round 2 is eight runs, all without subsampling, all at dim 100 unless stated:

| run | what it isolates |
|---|---|
| `w2_e3` | the anchor — the round-1 winner, now without subsampling |
| `w2_sub_e3` | the same thing *with* subsampling, to prove the drop at window 2 too |
| `w1_e3` | window 1: immediate neighbours only |
| `w3_e3` | window 3: is 2 a peak or just better than 5? |
| `w2_e5` | does a fifth pass help when the window is right? |
| `w2_k10_e3` | ten negatives at the winning window |
| `w2_k15_e3` | fifteen negatives — where does it stop paying? |
| `w2_d50_e3` | 50 dimensions: if big hurt, does small help? |

Every run is 50M sampled pairs per "epoch". Results go to `runs/<name>/`; a run whose
`metrics.json` exists is skipped, so a dead session can be resumed.

In [ ]:
CONFIGS = [
    {"name": "w2_e3", "dim": 100, "window": 2, "negatives": 5, "subsample": 0.0, "epochs": 3},
    {"name": "w2_sub_e3", "dim": 100, "window": 2, "negatives": 5, "subsample": 1e-4, "epochs": 3},
    {"name": "w1_e3", "dim": 100, "window": 1, "negatives": 5, "subsample": 0.0, "epochs": 3},
    {"name": "w3_e3", "dim": 100, "window": 3, "negatives": 5, "subsample": 0.0, "epochs": 3},
    {"name": "w2_e5", "dim": 100, "window": 2, "negatives": 5, "subsample": 0.0, "epochs": 5},
    {"name": "w2_k10_e3", "dim": 100, "window": 2, "negatives": 10, "subsample": 0.0, "epochs": 3},
    {"name": "w2_k15_e3", "dim": 100, "window": 2, "negatives": 15, "subsample": 0.0, "epochs": 3},
    {"name": "w2_d50_e3", "dim": 50, "window": 2, "negatives": 5, "subsample": 0.0, "epochs": 3},
]

In [ ]:
runs = Path("runs")
rows = []

for config in CONFIGS:
    run = runs / config["name"]
    metrics_path = run / "metrics.json"

    if metrics_path.exists():
        print(f"{config['name']}: already scored, skipping", flush=True)
        rows.append(json.loads(metrics_path.read_text(encoding="utf-8")))
        continue

    print(f"{config['name']}: dim={config['dim']} window={config['window']} "
          f"negatives={config['negatives']} subsample={config['subsample']:g} "
          f"epochs={config['epochs']}", flush=True)
    summary, embedding = train(config, ids, probabilities, len(vocab))
    metrics = {"run": config["name"], **config, **summary, **score(embedding, words)}

    run.mkdir(parents=True, exist_ok=True)
    np.save(run / "embeddings.npy", embedding)
    (run / "vocab.txt").write_text("\n".join(words), encoding="utf-8")
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    rows.append(metrics)
    print(f"  baseline {metrics['baseline']:.3f}  pc1 {metrics['pc1']:.3f}  "
          f"margin {metrics['group_margin']:+.3f}  analogies {metrics['analogies']}\n", flush=True)

table = pd.DataFrame(rows)[
    ["run", "dim", "window", "negatives", "subsample", "steps",
     "baseline", "pc1", "group_margin", "analogies", "median_rank", "seconds"]
]
table

## 9. Reading the table

- **Compare like with like.** `baseline` drifts between runs; a cosine of 0.7 means nothing until
  you know the floor that run started from.
- **`pc1` is a gate, not a score.** Above roughly 0.5 the space is collapsing and the other
  numbers are meaningless.
- **`group_margin`** is the headline: how much closer are words that belong together than two
  random words? It cannot be gamed by one lucky pair.
- **Loss is not in this table on purpose.** BCE averages over `K + 1` columns, so a run with
  `negatives=10` starts from a lower loss than one with `K=5` for arithmetic reasons, not
  because it learned more. Loss is in the training log; compare it only between runs with
  the same `K`.
- **Median rank** is the fine-grained analogue metric. Top-1 accuracy over 16 questions moves in
  steps of 6%; the median rank moves continuously.

The neighbours of all 25 display words are stored in each run's `metrics.json`.

In [ ]:
best = max(rows, key=lambda row: (row["group_margin"], row.get("accuracy", 0)))
print(f"best: {best['run']}  margin {best['group_margin']:+.3f}  "
      f"analogies {best['analogies']}  pc1 {best['pc1']:.3f}\n")

display_rows = []
for group, members in GROUPS.items():
    for word in members:
        found = best["neighbours"].get(word)
        if found:
            display_rows.append({
                "group": group,
                "word": word,
                "nearest": ", ".join(f"{w} {s:.2f}" for w, s in found),
            })
pd.DataFrame(display_rows)

## 10. What to do with the winner

Copy `runs/<name>/embeddings.npy` and `runs/<name>/vocab.txt` out of the notebook output and drop
them next to the project, replacing `outputs/`. The walkthrough notebook and the 3D map read
those two files and nothing else, so a better model needs no code changes.

A caution learned the hard way: more epochs is not automatically better on a 20M-token corpus.
Watch `group_margin` and `median_rank`, not the loss. The loss falls because the model gets
better at telling real pairs from random ones — a task it can improve at by memorising frequent
words.

### What the winner gets wrong

An aggregate hides the failures. These are the questions the best run still answers wrong —
usually the ones where the corpus itself is ambiguous, not where the arithmetic broke.

In [ ]:
misses = [row for row in best.get("analogies_detail", [])
          if row["gave"] != row["expected"]]
print(f"{len(misses)} misses out of {len(best.get('analogies_detail', []))}")
pd.DataFrame(misses) if misses else print("none")